In [ ]:
import rclpy
from rclpy.node import Node
from visualization_msgs.msg import Marker, MarkerArray
from std_msgs.msg import ColorRGBA
from geometry_msgs.msg import Point, Quaternion
import tf_transformations as tf
import numpy as np
import time

rclpy.init()
node = Node('circular')
pub = node.create_publisher(MarkerArray, '/marker_array', 1)

In [9]:
def point(theta, id, radius=1.0, scale=0.01, color=ColorRGBA(r=0., g=1., b=1., a=1.), frame_id='world', ns='circle'):
    m = Marker()
    m.ns = ns
    m.type = Marker.SPHERE
    m.id = id
    m.scale.x = m.scale.y = m.scale.z = scale
    m.color = color
    m.header.frame_id = frame_id

    q = tf.quaternion_about_axis(angle=theta, axis=[0,0,1])
    m.pose.orientation = Quaternion(x=q[0], y=q[1], z=q[2], w=q[3])
    p = tf.quaternion_matrix(q)[:3, :3].dot([1, 0, 0])
    m.pose.position = Point(x=p[0], y=p[1], z=p[2])
    return m

In [5]:
# create static array of points
points = [point(theta, id=i+10) for i, theta in enumerate(np.arange(0,2*np.pi,0.05))]
pub.publish(MarkerArray(markers=points))

In [15]:
# create moving point
theta = 0.0
color = ColorRGBA(r=1.0, g=0.0, b=0.0, a=1.0)

try:
    while rclpy.ok():
        theta += 0.05
        pub.publish(MarkerArray(markers=[point(theta, id=9, scale=0.02, color=color, ns='point')]))
        time.sleep(0.1)
except KeyboardInterrupt:
    pass